# NASA C-MAPSS Turbofan Engine RUL Prediction
## Notebook 03: Classical Machine Learning Models

### Objective
In this notebook, we train, validate, and evaluate classical regression models on engineered rolling features:
1. **Baseline**: Linear Regression (Ordinary Least Squares)
2. **Bagging Ensemble**: Random Forest Regressor
3. **Boosting Ensemble**: XGBoost Regressor
4. **Evaluation**: Compare RMSE, MAE, R², and the official NASA Asymmetric Score on 100 test engines.
5. **Feature Importance**: Analyze the physical drivers behind model predictions.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from config.config import MODELS_DIR, PROCESSED_DATA_DIR, RANDOM_SEED
from src.models import get_linear_regression, get_random_forest, get_xgboost
from src.evaluate import evaluate_predictions

print("Modules loaded successfully.")

### 1. Load Engineered Tabular Datasets
We load the 42-feature matrices (14 instantaneous, 14 rolling means, 14 rolling stds) generated in Stage 3.

In [ ]:
X_train = pd.read_parquet(PROCESSED_DATA_DIR / "FD001_X_train_tabular.parquet")
y_train = pd.read_parquet(PROCESSED_DATA_DIR / "FD001_y_train_tabular.parquet")["RUL"].values

X_val = pd.read_parquet(PROCESSED_DATA_DIR / "FD001_X_val_tabular.parquet")
y_val = pd.read_parquet(PROCESSED_DATA_DIR / "FD001_y_val_tabular.parquet")["RUL"].values

X_test = pd.read_parquet(PROCESSED_DATA_DIR / "FD001_X_test_tabular.parquet")
y_test = np.load(PROCESSED_DATA_DIR / "FD001_y_test_tabular.npy")

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}, y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

### 2. Train Models & Evaluate on Test Set
We evaluate each model using RMSE, MAE, R², and the NASA Asymmetric Scoring function.

In [ ]:
models = {
    "Linear Regression": get_linear_regression(),
    "Random Forest": get_random_forest(n_estimators=100, max_depth=12, random_state=RANDOM_SEED),
    "XGBoost": get_xgboost(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=RANDOM_SEED)
}

metrics_records = []
test_predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_test), 0, None)
    test_predictions[name] = pred
    m = evaluate_predictions(y_test, pred)
    m["Model"] = name
    metrics_records.append(m)

comparison_df = pd.DataFrame(metrics_records)[["Model", "RMSE", "MAE", "R2", "NASA_Score", "Early_Predictions", "Late_Predictions"]]
comparison_df

### 3. Feature Importance Analysis
We examine which sensor channels and rolling temporal statistics contribute most to decision boundaries.

In [ ]:
rf_model = models["Random Forest"]
xgb_model = models["XGBoost"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rf_top = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)
xgb_top = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(10)

sns.barplot(x=rf_top.values, y=rf_top.index, ax=axes[0], color="#2980b9")
axes[0].set_title("Random Forest: Top 10 Features (MDI)")

sns.barplot(x=xgb_top.values, y=xgb_top.index, ax=axes[1], color="#27ae60")
axes[1].set_title("XGBoost: Top 10 Features (Gain)")

plt.tight_layout()
plt.show()

### 4. Prediction vs Ground Truth Across 100 Test Engines
We compare predicted remaining cycles against ground-truth for all 100 test engines, sorted by actual lifespan.

In [ ]:
sort_idx = np.argsort(y_test)
y_sorted = y_test[sort_idx]

plt.figure(figsize=(12, 5))
plt.plot(np.arange(100), y_sorted, label="Actual Ground Truth RUL", color="black", linewidth=2, linestyle="--")

colors = {"Linear Regression": "gray", "Random Forest": "#2980b9", "XGBoost": "#27ae60"}
for name, pred in test_predictions.items():
    plt.plot(np.arange(100), pred[sort_idx], label=name, color=colors[name], alpha=0.8)

plt.title("Test Engine RUL Predictions vs Actual (Sorted by Ascending Actual RUL)")
plt.xlabel("Engine Index (Ascending RUL)")
plt.ylabel("Remaining Operational Cycles")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()